# **Routing Pharmaceutical Document Queries Using Metadata (No Embeddings Needed)**
In this notebook, you'll learn how to route user queries to the most relevant PDF pages using simple metadata—without relying on embeddings. We'll upload a real multi-page pharmaceutical document, extract text from each page, classify the document type, and use metadata filtering to quickly match queries like “What are the part numbers listed?” to relevant content such as certificates of quality or packaging specifications. This approach keeps retrieval fast, explainable, and cost-effective.



In [1]:
# ============================================================
# STEP 1: Install Required Libraries
# ============================================================
#
# WHAT WE'RE DOING:
# Installing LlamaIndex and its PDF reader for loading
# pharmaceutical blob files.
#
# WHY THIS MATTERS:
# LlamaIndex provides the document loading and retrieval
# framework we'll use throughout this notebook.
#
# WHAT YOU'LL SEE:
# Package installation output (may take 1-2 minutes).
# ============================================================

!pip install -q llama-index
!pip install -q llama-index-readers-file

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/nix/store/rkq033rbwmkdik3b9l8wmj0rxpph5hlc-python3-3.14.7-env/lib/python3.14/site-packages/filetype'
Check the permissions.

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/nix/store/rkq033rbwmkdik3b9l8wmj0rxpph5hlc-python3-3.14.7-env/lib/python3.14/site-packages/striprtf'
Check the permissions.



Link to the sample pharmaceutical document used in this notebook.


In [2]:
# ============================================================
# STEP 2: Load the Pharmaceutical Blob PDF
# ============================================================
#
# WHAT WE'RE DOING:
# Loading the 10-page pharmaceutical blob PDF using
# LlamaIndex's PDFReader, which returns one Document per page.
#
# WHY THIS MATTERS:
# Each page becomes a separate document object, giving us
# page-level control for classification and routing.
#
# WHAT YOU'LL SEE:
# The total number of pages loaded and a preview of the
# first page's text content.
# ============================================================

from llama_index.readers.file import PDFReader
import time

# Load the PDF
loader = PDFReader()
documents = loader.load_data("pharma-blob-sample.pdf")

# Preview to confirm it worked
print(f"Loaded {len(documents)} pages")
print(documents[0].text[:300])

Loaded 10 pages
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and


**Step 1: Store File-Level Metadata**

When users upload pharmaceutical PDFs, don't just save the raw file—attach useful metadata that helps identify what the document contains.

In [3]:
# ============================================================
# STEP 3: Store File-Level Metadata
# ============================================================
#
# WHAT WE'RE DOING:
# Attaching metadata to each page: file ID, user ID, year,
# filename, page number, and a placeholder doc_type.
#
# WHY THIS MATTERS:
# This metadata is what enables filtering later — we can
# route queries by document type, year, or source file.
#
# WHAT YOU'LL SEE:
# The metadata dictionary for the first page, showing all
# the fields we've attached.
# ============================================================

import uuid

file_id = str(uuid.uuid4())  # Assigns Unique ID
user_id = "xyz"
year = "2024"
filename = "pharma-blob-sample.pdf"

pdf_metadata_store = []

for i, doc in enumerate(documents):
    metadata = {
        "file_id": file_id,
        "user_id": user_id,
        "doc_type": "unknown",  # We'll classify it later
        "year": year,
        "filename": filename,
        "page_number": i + 1,
        "text": doc.text
    }
    pdf_metadata_store.append(metadata)

print("Stored Metadata:")
print(pdf_metadata_store[0])

Stored Metadata:
{'file_id': '3cb64b21-82c1-4d9f-a44b-54cdfec2d401', 'user_id': 'xyz', 'doc_type': 'unknown', 'year': '2024', 'filename': 'pharma-blob-sample.pdf', 'page_number': 1, 'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescript

**Step 2: Classify the User Query**

Before you search through documents, figure out what kind of pharmaceutical document the question is about. This helps you focus only on the most relevant files.

In [4]:
# ============================================================
# STEP 4: Set Up Gemini LLM Connection
# ============================================================
#
# WHAT WE'RE DOING:
# Creating a helper function that sends prompts to Google's
# Gemini 2.0 Flash model using your API key from Colab secrets.
#
# WHY THIS MATTERS:
# We'll use this function for both query classification and
# document type classification in the next steps.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================================

from llama_cpp import Llama
import os

# Download Mistral model if not already present
model_path = "content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")


def gemini_model(prompt):
    #response = llm(prompt, max_tokens=256, temperature=0.1)
    response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": prompt}
    ]
)
    return response['choices'][0]['message']['content']

ggml_metal_device_init: testing tensor API for f16 support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x101215d30 | th_max = 1024 | th_width =   32
ggml_metal_device_init: testing tensor API for bfloat support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x101215eb0 | th_max = 1024 | th_width =   32
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa_aux' library in 0.005 sec
ggml_metal_library_compile_all: compiled 'fa_f16' library in 0.058 sec
ggml_metal_library_compile_all: compiled 'fa_f32' library in 0.065 sec
ggml_metal_library_compile_all: compiled 'fa_q4_0' library in 0.004 sec
ggml_metal_library_compile_all: compiled 'fa_q4_1' library in 0.

Model file exists. Size: 4166.07 MB


create_tensor: loading tensor blk.7.ffn_norm.weight
create_tensor: loading tensor rope_freqs.weight
create_tensor: loading tensor blk.7.ffn_gate.weight
create_tensor: loading tensor blk.7.ffn_down.weight
create_tensor: loading tensor blk.7.ffn_up.weight
create_tensor: loading tensor blk.7.ffn_gate.bias
create_tensor: loading tensor blk.7.ffn_down.bias
create_tensor: loading tensor blk.7.ffn_up.bias
create_tensor: loading tensor blk.8.attn_norm.weight
create_tensor: loading tensor blk.8.attn_qkv.weight
create_tensor: loading tensor blk.8.attn_q.weight
create_tensor: loading tensor blk.8.attn_k.weight
create_tensor: loading tensor blk.8.attn_v.weight
create_tensor: loading tensor blk.8.attn_q.bias
create_tensor: loading tensor blk.8.attn_k.bias
create_tensor: loading tensor blk.8.attn_v.bias
create_tensor: loading tensor blk.8.attn_output.weight
create_tensor: loading tensor blk.8.attn_output.bias
create_tensor: loading tensor blk.8.ffn_norm.weight
create_tensor: loading tensor rope_freq

Model loaded successfully!


In [5]:
# ============================================================
# STEP 5: Classify User Query by Document Type
# ============================================================
#
# WHAT WE'RE DOING:
# Using the LLM to predict which pharmaceutical document type
# is most likely to answer the user's query.
#
# WHY THIS MATTERS:
# Instead of searching all pages, we narrow down to just the
# relevant document type first — faster and more accurate.
#
# WHAT YOU'LL SEE:
# The predicted document type (e.g., "packaging_specification").
# ============================================================

# Valid document type labels used throughout this notebook
VALID_DOC_TYPES = [
    "cover_letter", "certificate_of_quality", "packaging_specification",
    "bse_tse_declaration", "material_description", "supplier_qualification",
    "chain_of_custody", "unknown"
]

def clean_llm_label(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    # Check if the cleaned response contains a valid label
    for label in VALID_DOC_TYPES:
        if label in cleaned:
            return label
    return cleaned

def classify_query_llm(query, metadata_store):
    doc_list = "\n".join(
        [f"{i+1}. {doc['filename']} - doc_type: {doc['doc_type']}" for i, doc in enumerate(metadata_store)]
    )

    prompt = f"""
  You are an intelligent assistant that routes user queries to the most relevant pharmaceutical document.

  Available documents:
  {doc_list}

  User query: "{query}"

  Which document(s) are most likely to contain the answer?
  Respond with one of the following types:
  ["cover_letter", "certificate_of_quality", "packaging_specification", "bse_tse_declaration",
"material_description", "supplier_qualification", "chain_of_custody", "unknown"]

  Respond with ONLY the label. No explanation.
  """

    response = gemini_model(prompt)
    return clean_llm_label(response)

In [6]:
# ============================================================
# STEP 6: Test Query Classification
# ============================================================
#
# WHAT WE'RE DOING:
# Testing the classify_query_llm function with a sample
# pharmaceutical query about part numbers.
#
# WHY THIS MATTERS:
# We need to verify the LLM correctly identifies which
# document type should contain the answer.
#
# WHAT YOU'LL SEE:
# The predicted document type (e.g., "packaging_specification"
# or "certificate_of_quality").
# ============================================================

query = "What are the part numbers listed in this document?"
predicted_doc_type = classify_query_llm(query, pdf_metadata_store)
print(predicted_doc_type)

ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_bin_fuse_f32_f32_f32_4', name = 'kernel_bin_fuse_f32_f32_f32_4_op=0_nf=1_rb=1_cb=0'
ggml_metal_library_compile_pipeline: loaded kernel_bin_fuse_f32_f32_f32_4_op=0_nf=1_rb=1_cb=0      0x101219cb0 | th_max = 1024 | th_width =   32
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_rms_norm_mul_f32_4', name = 'kernel_rms_norm_mul_f32_4'
ggml_metal_library_compile_pipeline: loaded kernel_rms_norm_mul_f32_4                     0x10121a730 | th_max = 1024 | th_width =   32
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_mul_mv_q6_K_f32', name = 'kernel_mul_mv_q6_K_f32_nsg=2_ne12=1_r2=1_r3=1_split=0'
ggml_metal_library_compile_pipeline: loaded kernel_mul_mv_q6_K_f32_nsg=2_ne12=1_r2=1_r3=1_split=0      0x10121b6b0 | th_max = 1024 | th_width =   32
llama_perf_context_print:        load time =    4761.83 ms
llama_perf_context_print: prompt eval time =    4761.21 ms /   334 tokens 

material_description


Before moving on to Step 3, we first need to classify each PDF page based on its content and assign a doc_type using a simple rule-based function.



In [7]:
# ============================================================
# STEP 7: Classify Document Content
# ============================================================
#
# WHAT WE'RE DOING:
# Defining a function that classifies each page's content
# into one of the 7 pharmaceutical document types using the LLM.
#
# WHY THIS MATTERS:
# Once every page has a doc_type label, we can filter
# retrieval to only search within the right document type.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================================

def classify_doc_type_llm(text, max_chars=1000):
    truncated_text = text[:max_chars]

    prompt = f"""
You are classifying a pharmaceutical document into one of the following types:
["cover_letter", "certificate_of_quality", "packaging_specification", "bse_tse_declaration",
"material_description", "supplier_qualification", "chain_of_custody", "unknown"]

Document content:
\"\"\"
{truncated_text}
\"\"\"

What is the best doc_type label for this document? Respond with ONLY one of the labels above. No explanation.
"""
    try:
        response = gemini_model(prompt)
        return clean_llm_label(response)
    except Exception as e:
        print("LLM failed:", e)
        return "unknown"

In [8]:
# ============================================================
# STEP 8: Apply Document Classification to All Pages
# ============================================================
#
# WHAT WE'RE DOING:
# Running the classifier on every page to assign a doc_type
# label (e.g., "certificate_of_quality", "bse_tse_declaration").
#
# WHY THIS MATTERS:
# This is the labeling step — once complete, each page in
# our metadata store has a meaningful document type tag.
#
# WHAT YOU'LL SEE:
# Progress messages as each page is classified (10 pages total).
# ============================================================

for i, doc in enumerate(pdf_metadata_store):
    print(f"Classifying doc {i+1}...")
    doc["doc_type"] = classify_doc_type_llm(doc["text"])
    time.sleep(1)  # Avoid rate limiting

Llama.generate: 6 prefix-match hit, remaining 430 prompt tokens to eval


Classifying doc 1...


llama_perf_context_print:        load time =    4761.83 ms
llama_perf_context_print: prompt eval time =    3687.08 ms /   430 tokens (    8.57 ms per token,   116.62 tokens per second)
llama_perf_context_print:        eval time =     112.77 ms /     3 runs   (   37.59 ms per token,    26.60 tokens per second)
llama_perf_context_print:       total time =    3800.69 ms /   433 tokens
llama_perf_context_print:    graphs reused =          2


Classifying doc 2...


Llama.generate: 86 prefix-match hit, remaining 303 prompt tokens to eval
llama_perf_context_print:        load time =    4761.83 ms
llama_perf_context_print: prompt eval time =    2613.37 ms /   303 tokens (    8.62 ms per token,   115.94 tokens per second)
llama_perf_context_print:        eval time =     193.81 ms /     5 runs   (   38.76 ms per token,    25.80 tokens per second)
llama_perf_context_print:       total time =    2809.17 ms /   308 tokens
llama_perf_context_print:    graphs reused =          4


...

Llama.generate: 86 prefix-match hit, remaining 416 prompt tokens to eval
llama_perf_context_print:        load time =    4761.83 ms
llama_perf_context_print: prompt eval time =    3535.72 ms /   416 tokens (    8.50 ms per token,   117.66 tokens per second)
llama_perf_context_print:        eval time =     151.05 ms /     4 runs   (   37.76 ms per token,    26.48 tokens per second)
llama_perf_context_print:       total time =    3689.17 ms /   420 tokens
llama_perf_context_print:    graphs reused =          3


Classifying doc 9...


Llama.generate: 91 prefix-match hit, remaining 378 prompt tokens to eval
llama_perf_context_print:        load time =    4761.83 ms
llama_perf_context_print: prompt eval time =    3243.69 ms /   378 tokens (    8.58 ms per token,   116.53 tokens per second)
llama_perf_context_print:        eval time =     166.83 ms /     4 runs   (   41.71 ms per token,    23.98 tokens per second)
llama_perf_context_print:       total time =    3412.85 ms /   382 tokens
llama_perf_context_print:    graphs reused =          3


Classifying doc 10...


Llama.generate: 86 prefix-match hit, remaining 350 prompt tokens to eval
llama_perf_context_print:        load time =    4761.83 ms
llama_perf_context_print: prompt eval time =    3113.63 ms /   350 tokens (    8.90 ms per token,   112.41 tokens per second)
llama_perf_context_print:        eval time =     263.54 ms /     7 runs   (   37.65 ms per token,    26.56 tokens per second)
llama_perf_context_print:       total time =    3379.59 ms /   357 tokens
llama_perf_context_print:    graphs reused =          6


**Step 3: Retrieve Files Matching That Doc Type**

Once you've classified the query, use metadata to fetch only the relevant pharmaceutical documents.

In [9]:
# ============================================================
# STEP 9: Retrieve Files by Document Type
# ============================================================
#
# WHAT WE'RE DOING:
# Filtering the metadata store to return only pages whose
# doc_type matches the predicted type from the user's query.
#
# WHY THIS MATTERS:
# This is the routing payoff — instead of searching all 10
# pages, we only look at pages of the right document type.
#
# WHAT YOU'LL SEE:
# The matched documents with their metadata and text content.
# ============================================================

def retrieve_files_by_doc_type(doc_type, metadata_store):
    return [doc for doc in metadata_store if doc["doc_type"] == doc_type]

# Debug: show what we're matching against
print(f"Query predicted type: '{predicted_doc_type}'")
print(f"Page types in store: {[doc['doc_type'] for doc in pdf_metadata_store]}")
print()

# Call the function using the predicted type
matched_files = retrieve_files_by_doc_type(predicted_doc_type, pdf_metadata_store)

# Display matched documents
print(f"Found {len(matched_files)} matching page(s):")
for doc in matched_files:
    print(f"  Page {doc['page_number']}: {doc['doc_type']} - {doc['text'][:100]}...")

Query predicted type: 'material_description'
Page types in store: ['material_description', 'certificate_of_quality', 'certificate_of_quality', 'packaging_specification', 'packaging_specification', 'bse_tse_declaration', 'material_description', 'supplier_qualification', 'supplier_qualification', 'chain_of_custody']

Found 2 matching page(s):
  Page 1: material_description - Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AK...
  Page 7: material_description - Material Description Sheet
Product: AKTA ready Gradient Flow Section With Inlets
Part Number: 291846...


**Step 4 (Optional): Fall Back to Keyword-Based Search**

If your metadata filter still leaves too many results — or the user's query is vague — you can run a keyword-based search, but only within the smaller set of matching pharmaceutical documents.

In [10]:
# ============================================================
# STEP 10: Fallback — Keyword-Based Search
# ============================================================
#
# WHAT WE'RE DOING:
# If metadata filtering returns too many results or the query
# is vague, we can narrow further with keyword matching.
#
# WHY THIS MATTERS:
# Keyword search is a simple fallback that works well for
# pharmaceutical terms like "part number", "lot", "BSE".
#
# WHAT YOU'LL SEE:
# The best matching document from the keyword search.
# ============================================================

def keyword_search(query, documents):
    query = query.lower()
    keywords = ["part number", "article number", "item", "specification",
                "product", "certificate", "lot", "description",
                "sterilization", "BSE", "supplier"]

    for doc in documents:
        text = doc["text"].lower()
        if any(keyword.lower() in text for keyword in keywords):
            return doc
    return None

user_query = "What part numbers are listed?"
print("Best matching result (fallback):")
print(keyword_search(user_query, matched_files))

Best matching result (fallback):
{'file_id': '3cb64b21-82c1-4d9f-a44b-54cdfec2d401', 'user_id': 'xyz', 'doc_type': 'material_description', 'year': '2024', 'filename': 'pharma-blob-sample.pdf', 'page_number': 1, 'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during 